# 07 clv modeling

Check finite-horizon forecasts against observed revenue and a simple extrapolation; assumptions may fail.

Reusable calculations live in src/ and run_pipeline.py. This notebook reads executed artifacts so analysis cannot silently retrain or leak future data.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
results=json.loads((root/'artifacts/tables/results.json').read_text())

In [2]:
data=pd.read_csv(root/'artifacts/tables/customer_scores.csv')
display(data.head(12))
display(data.describe(include='all').T)

,customer,first,last,frequency,monetary,recency,tenure,aov,recent_orders,inactive,cutoff,risk,future_revenue,simple_revenue,expected_purchases,expected_order_value,observed_future_revenue,priority,segment
0,12766,2010-10-28 09:29:00,2011-01-09 15:56:00,4,3765.13,29.336111,102.604861,941.282500,3,0,2011-02-08,0.324726,1363.382430,2814.708835,1.671800,815.517463,2179.94,442.725787,High value / Low risk
1,15379,2010-11-04 11:59:00,2011-01-20 14:01:00,4,3856.64,18.415972,95.500694,964.160000,3,0,2011-02-08,0.320845,1335.661033,3093.346667,1.746455,764.784175,371.84,428.539626,High value / Low risk
2,12578,2010-09-22 11:19:00,2011-01-13 15:34:00,3,3456.60,25.351389,138.528472,1152.200000,2,1,2011-02-08,0.436307,942.726873,1914.807194,1.009129,934.198715,0.00,411.318736,High value / Low risk
3,17404,2010-01-07 11:31:00,2011-01-28 10:00:00,11,22647.36,10.583333,396.520139,2058.850909,3,0,2011-02-08,0.110878,3690.961136,4392.561008,1.776855,2077.244257,4522.68,409.246765,High value / Low risk
4,14016,2010-01-08 14:12:00,2010-12-17 14:21:00,8,7815.94,52.402083,395.408333,976.992500,1,0,2011-02-08,0.319474,1118.500333,1519.766111,1.280082,873.772619,1477.56,357.331783,High value / Low risk
5,13725,2011-02-04 12:18:00,2011-02-07 15:14:00,2,1168.63,0.365278,3.487500,584.315000,2,0,2011-02-08,0.594187,585.715988,22496.127500,1.807879,323.979582,430.17,348.024547,High value / High risk
6,12476,2010-07-26 09:04:00,2011-01-31 11:46:00,14,7829.52,7.509722,196.622222,559.251429,2,0,2011-02-08,0.169730,1974.715416,2185.906599,2.828147,698.236494,1571.87,335.168934,High value / Low risk
7,17284,2010-09-20 16:18:00,2011-01-07 14:31:00,3,1856.69,31.395139,140.320833,618.896667,2,0,2011-02-08,0.488659,670.641198,1013.937092,0.998557,671.610345,207.24,327.715077,High value / High risk
8,17204,2010-06-10 11:37:00,2010-11-30 17:58:00,5,3940.13,69.251389,242.515972,788.026000,2,0,2011-02-08,0.353279,914.736885,998.814848,0.916698,997.861183,1250.16,323.157510,High value / Low risk
9,13615,2010-01-28 15:43:00,2011-01-25 08:21:00,8,6272.53,13.652083,375.345139,784.066250,1,1,2011-02-08,0.327733,979.243052,1284.534069,1.340214,730.661497,0.00,320.930255,High value / Low risk


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer,363.0,NaN,NaN,NaN,15341.253444,1778.66688,12352.0,13769.0,15326.0,17069.5,18269.0
first,363,362,2009-12-07 12:19:00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last,363,359,2010-12-21 14:26:00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
frequency,363.0,NaN,NaN,NaN,8.00551,14.609464,1.0,2.0,5.0,9.0,166.0
monetary,363.0,NaN,NaN,NaN,3416.926612,10009.559136,51.41,523.245,1410.18,3183.27,138699.89
recency,363.0,NaN,NaN,NaN,39.222549,26.704009,0.31875,12.476042,32.625,65.456944,76.580556
tenure,363.0,NaN,NaN,NaN,273.162628,149.361175,0.48125,124.463194,316.476389,421.423611,433.620833
aov,363.0,NaN,NaN,NaN,357.068946,305.325336,44.17625,196.14119,291.86,406.879444,3590.59
recent_orders,363.0,NaN,NaN,NaN,2.454545,3.922594,1.0,1.0,2.0,3.0,57.0
inactive,363.0,NaN,NaN,NaN,0.517906,0.500369,0.0,0.0,1.0,1.0,1.0


Interpretation: consult the corresponding report for mathematical definitions, executed estimates and their limits. Never turn an assumed campaign response into an observed result.

In [3]:
display(results['value'])

{'spearman_r': 0.22243985125024268,
 'p_value': 3.770631747033588e-34,
 'repeat_customers': 2928,
 'bg_parameters': {'r': 0.7385453171127513,
  'alpha': 68.39415707791873,
  'a': 0.009200399700005092,
  'b': 0.30594446535083714},
 'gg_parameters': {'p': 2.2561922836607904,
  'q': 3.5117641147880474,
  'v': 463.67818800651486},
 'horizon_days': 77,
 'predicted_total_test': 227263.28698695483,
 'observed_total_test': 167977.61,
 'mae': 420.38084472936595,
 'simple_mae': 744.9106912638299,
 'median_prediction': 327.4864038874745}

# Finite-horizon customer value
BG/NBD uses daily purchase occasions: frequency = repeat purchase days excluding the first day, recency = last minus first day, T = calibration end minus first day. Multiple invoices on one day are combined, matching the day time unit. Gamma-Gamma is fitted only to repeat buyers with positive mean repeat-day value; first occasions are excluded by the summary utility. Zero-repeat buyers receive population-shrunk monetary estimates. Estimated 77-day future gross revenue = expected future occasions × expected revenue per occasion. The un-discounted finite horizon avoids an unsupported perpetual lifetime claim; the benchmark extrapolates historical invoice intensity and average invoice value.
```json
{
  "spearman_r": 0.22243985125024268,
  "p_value": 3.770631747033588e-34,
  "repeat_customers": 2928,
  "bg_parameters": {
    "r": 0.7385453171127513,
    "alpha": 68.39415707791873,
    "a": 0.009200399700005092,
    "b": 0.30594446535083714
  },
  "gg_parameters": {
    "p": 2.2561922836607904,
    "q": 3.5117641147880474,
    "v": 463.67818800651486
  },
  "horizon_days": 77,
  "predicted_total_test": 227263.28698695483,
  "observed_total_test": 167977.61,
  "mae": 420.38084472936595,
  "simple_mae": 744.9106912638299,
  "median_prediction": 327.4864038874745
}
```
The Spearman diagnostic is descriptive. A nonzero association weakens independence; statistical nonsignificance would not prove it. Forecast MAE and total forecast/actual are measured on the held-out customer/time block. BG/NBD and Gamma-Gamma are fit using all customers' pre-test histories, which is legitimate unsupervised deployment information, but customer-disjoint supervised evaluation does not imply independent value-model customers. Value predictions are not guaranteed superior to the simple benchmark; report both. No forecast clipping hides tail error.

Numerical implementation: BG/NBD uses a 0.01 penalizer; Gamma-Gamma is unpenalized with q>1 enforced for a finite population mean. A penalized Gamma-Gamma fit reached the q=1 boundary and was rejected. For fitted BG/NBD a<1, the library's log-hypergeometric expectation produced invalid values for some zero-repeat histories. The equivalent finite-horizon expectation is integrated using 96-node Gauss-Jacobi quadrature over posterior dropout probability: conditional on alive, lambda~Gamma(r+x, alpha+T), p~Beta(a,b+x), and E[N(h)|lambda,p]=(1-exp(-lambda*p*h))/p. Multiply the integrated expectation by posterior probability alive. Tests compare it to the closed form in the stable parameter region and check finiteness/monotonicity for low-dropout fits. Saved value models contain parameters rather than unpickleable fitted lambda functions.

Primary model sources: [Fader/Hardie customer-base tutorial](https://www.brucehardie.com/talks/ho_cba_tut_art_14.pdf), [Gamma-Gamma formulation](https://www.brucehardie.com/notes/025/gamma_gamma.pdf).
